# Can a small bank-table network find defaults better than doing nothing?

A bank wants to prioritize credit-card accounts for further review rather than simply predict no default for everyone. Using the historical UCI Default of Credit Card Clients table through OpenML, you will build two small Keras candidates, select using validation loss, and report a same-test comparison. You will also save and reload the complete prediction path without turning a classroom threshold into an automatic credit decision.

**How this lab works.** Section numbers match the lesson page. In each section: **Run** the cells, **Predict** before you run (write your guess in the `guess(...)` cell), **Modify** one thing and see what changes, then **Make** the small piece marked `# YOUR CODE`. A ✓ means it works. The last cell prints a hand-in line for the lesson page.

In [ ]:
# 0. Setup — run this first (Runtime → Run all also works)
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import json, base64
import numpy as np
import keras
import tensorflow as tf

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| GPU:", bool(tf.config.list_physical_devices("GPU")))
if not keras.__version__.startswith("3."):
    raise RuntimeError("This lab needs Keras 3. In Colab: Runtime → Restart session, then run again.")
keras.utils.set_random_seed(42)      # same data split and starting weights every run

LESSON = "m1-7-keras-workflow-your-first"
_RESULTS, _GUESSES = {}, {}

def guess(key, value):
    """Write your prediction BEFORE running the next cell. Changing it afterwards is allowed but recorded."""
    _GUESSES[key] = value
    print(f"Guess saved: {key} = {value!r}. Now run the next cell and compare.")

def record(key, value):
    """Store a result for the hand-in at the end."""
    _RESULTS[key] = round(float(value), 6)
    return value

def check(name, ok, hint=""):
    """✓ when your code does what the section asks, ✗ with a hint when it does not."""
    print(("✓ " if ok else "✗ ") + name + ("" if ok else " — " + hint))
    return bool(ok)

def handin():
    payload = {"v": 1, "lesson": LESSON, "results": _RESULTS, "guesses": _GUESSES}
    code = base64.urlsafe_b64encode(json.dumps(payload, separators=(",", ":")).encode()).decode()
    print("Copy this whole line into the lesson page (Debrief → Hand-in):")
    print("HANDIN-1." + code)

## 1. 1. What would count as finding defaults?

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

data = fetch_openml(data_id=42477, as_frame=True, parser="auto")
features = data.data.copy()
identifier_columns = [c for c in features.columns if c.lower() == "id"]
features = features.drop(columns=identifier_columns)
raw_target = data.target.astype(str)
print("Predictor names:", features.columns.tolist())
print("Target name:", data.target.name)
print("Observed target values and counts:\n", raw_target.value_counts())
print("Actual first client, not a synthetic example:\n", data.frame.iloc[0])
print(f"Observed table: {len(features):,} clients and {features.shape[1]} predictors; expected 30,000 and 23.")

The UCI documentation defines the original target as **0 = no default, 1 = default**; this OpenML version presents those values as labels. Inspect the printed values before applying that explicit mapping—alphabetical category order is not a definition of default. Finish each **Make** cell before continuing: its unfinished starter is meant to fail its check.

In [ ]:
# Can always predicting no default be right on most clients yet detect no defaults?
# Enter the whole-table accuracy you expect, between 0 and 1.
guess("whole_table_majority_accuracy", 0.5)

In [ ]:
# YOUR CODE: explicitly map the documented no-default and default labels to 0 and 1.
target_mapping = None
y = raw_target.map(target_mapping).to_numpy(dtype="int32") if target_mapping is not None else None
whole_accuracy = float((y == 0).mean()) if y is not None else None
if y is not None:
    assert set(raw_target.unique()) == {"0", "1"}
    assert features.shape == (30000, 23)
    assert int(y.sum()) == 6636 and int((y == 0).sum()) == 23364
    print(f"Whole-table majority accuracy {whole_accuracy:.4f} (range 0–1); default recall 0 (range 0–1).")
    print("Detected defaults: 0 of 6,636; accuracy alone hides this failure.")
    record("whole_table_majority_accuracy", whole_accuracy)
check("target_and_majority", y is not None and set(np.unique(y)) == {0, 1} and abs(whole_accuracy - 0.7788) < 1e-12, "Map documented label '0' to 0 and label '1' to 1; majority accuracy is the share with y == 0.")

In [ ]:
# Try row 100 instead. Inspect its fields; do not change the target mapping.
row_number = 0
print(features.iloc[row_number])
print(f"Recorded default indicator: {y[row_number]} (0 = no default; 1 = default).")

## 2. 2. Split before learning the preparation.

In [ ]:
X = features.to_numpy(dtype="float32")
x_development, x_test, y_development, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=17)
x_train, x_val, y_train, y_val = train_test_split(x_development, y_development, test_size=0.25, stratify=y_development, random_state=17)
assert (len(x_train), len(x_val), len(x_test)) == (18000, 6000, 6000)
print("Train / validation / test shapes:", x_train.shape, x_val.shape, x_test.shape)
print("Reference sizes: 18,000 / 6,000 / 6,000 rows; each has 23 float32 predictors.")

Test was reserved first; validation is one quarter of the remaining rows. As in section 2, even preparation must learn from training inputs only: including reserved inputs in `adapt` would let them influence the prediction path without using their targets. Keeping this layer inside the model will also keep its reference statistics inside the saved artifact.

In [ ]:
norm = keras.layers.Normalization(axis=-1, name="training_normalization")
# YOUR CODE: learn normalization statistics from x_train only.
# YOUR CODE: check that all three input arrays are two-dimensional with 23 columns.
widths_ok = None
print("Feature widths match the 23-predictor contract:", widths_ok)
check("training_only_preparation", widths_ok and norm.built and np.allclose(norm.mean.numpy().ravel(), x_train.mean(axis=0), rtol=1e-5, atol=1e-4), "Assert 23 columns in every split and adapt normalization on x_train only, not the full table.")

In [ ]:
training_prevalence = float(y_train.mean())
def baseline_loss(target):
    return float(-np.mean(target * np.log(training_prevalence) + (1 - target) * np.log1p(-training_prevalence)))
val_baseline_loss = baseline_loss(y_val)
val_baseline_accuracy = float((y_val == 0).mean())
print(f"Training default prevalence {training_prevalence:.4f} (probability range 0–1).")
print(f"Validation baseline BCE {val_baseline_loss:.4f} vs uniform-probability BCE {np.log(2):.4f}; lower is better, minimum 0.")
print(f"Validation majority accuracy {val_baseline_accuracy:.4f}; default recall 0 (both range 0–1).")
record("validation_baseline_loss", val_baseline_loss)

In [ ]:
# Try another training row. Its values change; the fitted reference statistics must not.
training_row = 0
prepared_row = norm(x_train[training_row:training_row + 1]).numpy()[0]
print("First six raw fields:", x_train[training_row, :6])
print("First six normalized fields:", np.round(prepared_row[:6], 3))
print("Normalized reference: training mean approximately 0, standard deviation approximately 1 per nonconstant feature.")

## 3. 3. Build the network and its learning contract.

Lay out the bank network before building it: **23 raw inputs → fixed training normalization → 16 ReLU units → 1 sigmoid probability**. The Dense layers learn `(23 × 16 + 16) + (16 × 1 + 1) = 401` weights and biases; normalization keeps separate non-trainable state. Both experimental candidates will use the same frozen normalization layer, which `fit` does not update.

In [ ]:
# How many trainable Dense weights and biases does 23 → 16 → 1 contain?
guess("dense_parameters", 0)
# Does compile itself change the Dense weights? Enter yes or no.
guess("compile_updates_weights", "yes")

In [ ]:
def build_candidate(drop_rate=0.0):
    inputs = keras.Input(shape=(23,), name="borrower")
    prepared = norm(inputs)
    hidden = keras.layers.Dense(16, activation="relu", name="hidden")(prepared)
    if drop_rate: hidden = keras.layers.Dropout(drop_rate, seed=17, name="dropout")(hidden)
    # YOUR CODE: create a one-unit sigmoid Dense output, fed with hidden.
    outputs = None
    candidate = keras.Model(inputs, outputs) if outputs is not None else None
    if candidate is None: return None
    candidate.compile(optimizer=keras.optimizers.Adam(0.001), loss=keras.losses.BinaryCrossentropy(from_logits=False), metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5), keras.metrics.Recall(name="default_recall", thresholds=0.5)])
    return candidate
keras.utils.set_random_seed(17)
model = build_candidate()
check("probability_network", model is not None and sum(int(np.prod(w.shape)) for w in model.trainable_weights) == 401 and isinstance(model.loss, keras.losses.BinaryCrossentropy), "Feed hidden into a one-unit sigmoid Dense layer; its probabilities match BinaryCrossentropy(from_logits=False).")

In [ ]:
initial_dense = {name: [a.copy() for a in model.get_layer(name).get_weights()] for name in ("hidden", "default_probability")}
trainable_count = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
nontrainable_count = sum(int(np.prod(w.shape)) for w in model.non_trainable_weights)
model.summary()
print(f"Trainable count {trainable_count} vs hand calculation 401; normalization state {nontrainable_count} vs expected 47.")
model.compile(optimizer=keras.optimizers.Adam(0.001), loss=keras.losses.BinaryCrossentropy(from_logits=False), metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5), keras.metrics.Recall(name="default_recall", thresholds=0.5)])
compile_change = max(float(np.max(np.abs(a - b))) for name in initial_dense for a, b in zip(initial_dense[name], model.get_layer(name).get_weights()))
print(f"Largest Dense weight change caused by compile: {compile_change:.1f} (no-update reference 0).")
record("dense_parameters", trainable_count)
record("compile_weight_change", compile_change)

In [ ]:
toy_inputs = keras.Input(shape=(2,))
toy_prepared = keras.layers.Normalization(mean=[150000.0, 0.0], variance=[100000.0**2, 1.0])(toy_inputs)
toy_hidden = keras.layers.Dense(2, activation="relu", name="toy_hidden")(toy_prepared)
toy_outputs = keras.layers.Dense(1, activation="sigmoid", name="toy_output")(toy_hidden)
toy = keras.Model(toy_inputs, toy_outputs)
toy.get_layer("toy_hidden").set_weights([np.array([[0.8, -0.6], [0.4, 0.2]], dtype="float32"), np.array([-0.2, 0.1], dtype="float32")])
toy.get_layer("toy_output").set_weights([np.array([[1.2], [-0.5]], dtype="float32"), np.array([-0.4], dtype="float32")])
toy_borrower = np.array([[200000.0, 2.0]], dtype="float32")
toy_probability = float(toy(toy_borrower, training=False).numpy()[0, 0])
toy_loss = float(keras.losses.BinaryCrossentropy()(np.array([[1.0]], dtype="float32"), toy(toy_borrower, training=False)))
print(f"Miniature probability {toy_probability:.4f} vs trace 0.6682; recorded-default BCE {toy_loss:.4f} vs trace 0.4032 and uniform 0.6931.")
np.testing.assert_allclose([toy_probability, toy_loss], [0.66818777, 0.40318605], atol=1e-6)

In [ ]:
# Try 250000. Keep delay, reference statistics and weights fixed: this is not retraining.
toy_credit_limit = 200000.0
changed_borrower = np.array([[toy_credit_limit, 2.0]], dtype="float32")
changed_probability = float(toy(changed_borrower, training=False).numpy()[0, 0])
print(f"Changed-input probability {changed_probability:.4f} vs original {toy_probability:.4f}; probability range 0–1.")

## 4. 4. Fit means running the learning loop.

We define section 5's callback before the first `fit`, because `fit` needs the object now. The experiment resets random seeds to 17 immediately before each candidate's fit; the later candidate also receives copies of the original Dense arrays, a fresh Adam optimizer and a fresh callback. This is a controlled classroom comparison, not a claim that the two conditions consume identical random numbers.

In [ ]:
# Must the epoch with the lowest training loss also have the lowest validation loss?
guess("same_best_epoch", "yes")

In [ ]:
def fresh_stop():
    return keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", min_delta=0, patience=3, restore_best_weights=True)
early_stop = fresh_stop()
keras.utils.set_random_seed(17)
# YOUR CODE: call model.fit with x_train, y_train; batch_size=128; epochs=30;
# validation_data=(x_val, y_val); callbacks=[early_stop]; verbose=0.
history = None
if history is not None:
    print(f"Executed {len(history.epoch)} epochs vs cap 30; last recorded validation BCE {history.history['val_loss'][-1]:.4f} vs baseline {val_baseline_loss:.4f}.")
check("explicit_validation_fit", history is not None and "val_loss" in history.history and len(history.history["loss"]) <= 30, "Call model.fit with training arrays, explicit validation_data, batch size 128, cap 30 and callbacks=[early_stop].")

In [ ]:
# Try showing just the first 10 epochs. Change the view, not the fitted model.
show_epochs = 30
shown = min(show_epochs, len(history.epoch))
epochs = np.arange(1, shown + 1)
plt.plot(epochs, history.history["loss"][:shown], label="training")
plt.plot(epochs, history.history["val_loss"][:shown], label="validation")
plt.axhline(val_baseline_loss, linestyle="--", label="same-validation prevalence baseline")
plt.xlabel("Epoch (starting at 1)")
plt.ylabel("Binary cross-entropy (minimum 0; no finite upper bound)")
plt.legend(); plt.show()
print(f"Epoch {shown}: training BCE {history.history['loss'][shown - 1]:.4f}; validation BCE {history.history['val_loss'][shown - 1]:.4f} vs baseline {val_baseline_loss:.4f}.")

## 5. 5. Keep the best validation weights, not simply the last weights.

In [ ]:
# Practice history only: 0.56, 0.53, 0.51, 0.515, 0.52, 0.525.
# With patience 2 and min_delta 0, at which one-based epoch would training stop?
guess("practice_stopping_epoch", 0)

In [ ]:
plain_best_epoch = int(np.argmin(history.history["val_loss"])) + 1
plain_best_loss = float(min(history.history["val_loss"]))
plain_end_epoch = len(history.epoch)
plain_report = model.evaluate(x_val, y_val, verbose=0, return_dict=True)
print(f"Best validation epoch {plain_best_epoch}; run ended at {plain_end_epoch} of at most 30 epochs.")
print("Early-stopping event:", early_stop.stopped_epoch + 1 if early_stop.stopped_epoch else "none: epoch cap reached")
print(f"Restored validation BCE {plain_report['loss']:.4f} vs recorded best {plain_best_loss:.4f} and baseline {val_baseline_loss:.4f}.")
print(f"Restored validation accuracy {plain_report['accuracy']:.4f} vs majority {val_baseline_accuracy:.4f}; recall {plain_report['default_recall']:.4f} vs 0.")
np.testing.assert_allclose(plain_report["loss"], plain_best_loss, atol=1e-5)
record("plain_best_epoch", plain_best_epoch)
record("plain_end_epoch", plain_end_epoch)

In [ ]:
# Try patience 3 on this illustrative history only; do not refit either bank candidate.
practice_patience = 2
practice_losses = [0.56, 0.53, 0.51, 0.515, 0.52, 0.525]
practice_callback = keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", min_delta=0, patience=practice_patience, restore_best_weights=True)
practice_model = keras.Model(toy.input, toy.output)
practice_callback.set_model(practice_model)
practice_model.stop_training = False
practice_callback.on_train_begin()
for epoch, loss in enumerate(practice_losses):
    practice_callback.on_epoch_end(epoch, {"val_loss": loss})
    if practice_model.stop_training: break
practice_callback.on_train_end()
print(f"Practice ended at epoch {epoch + 1} of 6 supplied values; best epoch 3, BCE 0.51 vs initial 0.56.")
if practice_patience == 2: record("practice_stopping_epoch", epoch + 1)

The measured validation evaluation checks the weights currently loaded, not merely a number stored in `History`. With restoration enabled, those weights come from the best monitored epoch, whether training stopped early or reached its cap. Restoring model weights does not rewind Adam to its historical state at that epoch, so the next experiment starts with a fresh optimizer rather than another `fit` on this model.

## 6. 6. Dropout is a training condition to test.

The only second condition is **23 inputs → the same normalization → 16 ReLU units → Dropout(0.2) → 1 sigmoid output**. Splits, original Dense arrays, learning rate, batch size, epoch cap and stopping rule stay fixed; only the training condition changes. Compare restored validation metrics and select lower validation binary cross-entropy, with ties going to no dropout—not whichever condition has the better training loss.

In [ ]:
# Must dropout improve validation loss? Enter yes or no before seeing the comparison.
guess("dropout_must_help", "yes")

In [ ]:
keras.utils.set_random_seed(17)
drop_model = build_candidate(drop_rate=0.2)
for name, arrays in initial_dense.items():
    drop_model.get_layer(name).set_weights([a.copy() for a in arrays])
assert all(np.array_equal(a, b) for name in initial_dense for a, b in zip(initial_dense[name], drop_model.get_layer(name).get_weights()))
drop_model.summary()
drop_stop = fresh_stop()
keras.utils.set_random_seed(17)
drop_history = drop_model.fit(x_train, y_train, batch_size=128, epochs=30, validation_data=(x_val, y_val), callbacks=[drop_stop], verbose=0)
drop_report = drop_model.evaluate(x_val, y_val, verbose=0, return_dict=True)
drop_best_epoch = int(np.argmin(drop_history.history["val_loss"])) + 1
print(f"Dropout 0.2: best epoch {drop_best_epoch} of {len(drop_history.epoch)} executed, cap 30; restored validation BCE {drop_report['loss']:.4f} vs baseline {val_baseline_loss:.4f}.")

In [ ]:
for name, h, report in [("No dropout", history, plain_report), ("Dropout 0.2", drop_history, drop_report)]:
    best_epoch = int(np.argmin(h.history["val_loss"])) + 1
    print(f"{name}: best epoch {best_epoch}/{len(h.epoch)}; best validation BCE {min(h.history['val_loss']):.4f} vs baseline {val_baseline_loss:.4f}.")
    print(f"  Restored BCE {report['loss']:.4f}; accuracy {report['accuracy']:.4f} vs {val_baseline_accuracy:.4f}; default recall {report['default_recall']:.4f} vs 0.")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, name, h in zip(axes, ["No dropout", "Dropout 0.2"], [history, drop_history]):
    e = np.arange(1, len(h.epoch) + 1)
    ax.plot(e, h.history["loss"], label="training")
    ax.plot(e, h.history["val_loss"], label="validation")
    ax.axhline(val_baseline_loss, linestyle="--", label="validation baseline")
    ax.set(title=name, xlabel="Epoch", ylabel="BCE: minimum 0, no finite upper bound")
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# YOUR CODE: choose dropout only if its restored validation BCE is strictly lower.
use_dropout = None
selected_model = drop_model if use_dropout else model
selected_validation = drop_report if use_dropout else plain_report
selected_history = drop_history if use_dropout else history
print(f"Frozen choice: {'dropout 0.2' if use_dropout else 'no dropout'}; validation BCE {selected_validation['loss']:.4f} vs baseline {val_baseline_loss:.4f}.")
record("selected_validation_loss", selected_validation["loss"])
record("selected_dropout_rate", 0.2 if use_dropout else 0.0)
check("validation_only_selection", use_dropout is not None and bool(use_dropout) == bool(drop_report["loss"] < plain_report["loss"]), "Compare restored validation BCE with strict <; equality must choose no dropout. Do not use test scores.")

In [ ]:
# Try [0, 1] instead. This explicit toy mask is not a new bank-model condition.
mask = np.array([1.0, 0.0], dtype="float32")
activations = np.array([1.0, 0.2], dtype="float32")
scaled = activations * mask / (1 - 0.5)
masked_logit = float(scaled @ np.array([1.2, -0.5]) - 0.4)
masked_probability = float(1 / (1 + np.exp(-masked_logit)))
print("Scaled activations:", scaled, "vs unmasked", activations)
print(f"Specified-mask probability {masked_probability:.4f} vs unmasked {toy_probability:.4f}; default BCE {-np.log(masked_probability):.4f} vs unmasked {toy_loss:.4f}.")
probe = keras.layers.Dropout(0.5, seed=17)
np.testing.assert_allclose(probe(activations[None, :], training=False).numpy(), activations[None, :])
p1 = drop_model(x_val[:16], training=False).numpy()
p2 = drop_model(x_val[:16], training=False).numpy()
np.testing.assert_allclose(p1, p2, atol=1e-7)
print(f"Repeated bank-model inference maximum difference {np.max(np.abs(p1 - p2)):.1f} vs deterministic reference 0; dropout is inactive.")

## 7. 7. Freeze, evaluate and save.

The candidate choice is now frozen: open the test branch once, not once per candidate. Compare its loss with a constant **training-prevalence** probability on these test rows, and its hard predictions with always-no-default on these same rows. Recall divides detected defaults by actual defaults; neither accuracy nor a favorable miniature mask establishes lending readiness.

In [ ]:
# Will saving and reloading change ordinary inference probabilities on a fixed sample?
# Enter the maximum absolute difference you expect; zero means identical.
guess("reload_difference", 0.1)

In [ ]:
# YOUR CODE: evaluate the already-selected model on test once, returning a dictionary.
test_report = None
test_baseline_loss = baseline_loss(y_test)
test_baseline_accuracy = float((y_test == 0).mean())
test_baseline_recall = 0.0
check("frozen_test_evaluation", test_report is not None and {"loss", "accuracy", "default_recall"}.issubset(test_report), "Evaluate only selected_model on x_test, y_test with return_dict=True; do not compare candidates on test.")

In [ ]:
print("FINAL REPORT — fixed threshold 0.5; accuracy and recall range 0–1")
print(f"Selected validation BCE {selected_validation['loss']:.4f} vs same-validation baseline {val_baseline_loss:.4f}.")
print(f"Test BCE {test_report['loss']:.4f} vs same-test prevalence baseline {test_baseline_loss:.4f}; improved: {test_report['loss'] < test_baseline_loss}.")
print(f"Test accuracy {test_report['accuracy']:.4f} vs same-test majority {test_baseline_accuracy:.4f}; improved: {test_report['accuracy'] > test_baseline_accuracy}.")
print(f"Test default recall {test_report['default_recall']:.4f} vs same-test majority recall 0; improved: {test_report['default_recall'] > 0}.")
print("BCE has minimum 0 and no finite upper bound; lower is better. Review signal only, not an automatic credit decision.")
record("test_baseline_accuracy", test_baseline_accuracy)
record("test_baseline_loss", test_baseline_loss)
record("test_baseline_recall", test_baseline_recall)
record("test_loss", test_report["loss"])
record("test_accuracy", test_report["accuracy"])
record("test_default_recall", test_report["default_recall"])

In [ ]:
fixed_validation_sample = x_val[:32].copy()
save_path = "bank_review_model.keras"
before_reload = selected_model.predict(fixed_validation_sample, verbose=0)
selected_model.save(save_path)
reloaded_model = keras.models.load_model(save_path)
after_reload = reloaded_model.predict(fixed_validation_sample, verbose=0)
np.testing.assert_allclose(before_reload, after_reload, rtol=1e-6, atol=1e-7)
reload_difference = float(np.max(np.abs(before_reload - after_reload)))
print(f"Reload maximum probability difference {reload_difference:.2e} vs tolerance 1e-6.")
print("Saved architecture, Dense weights, normalization state and compiled optimizer state; ordinary inference uses no targets.")
record("reload_difference", reload_difference)

In [ ]:
# Try another validation borrower. Change inputs, not the model, normalization or threshold.
inference_row = 0
borrower = x_val[inference_row:inference_row + 1]
probability = float(reloaded_model(borrower, training=False).numpy()[0, 0])
print(f"Reloaded probability {probability:.4f} (range 0–1); classroom flag at strictly > 0.5: {probability > 0.5}.")
print("A review-prioritization illustration, not an automatic lending instruction or a calibrated-risk guarantee.")

Debug this proposed workflow: adapt on all clients, call `fit` again on the old compiled model for a 'fresh' comparison, and choose dropout using the better test score. Each step breaks a different boundary; editing the notebook afterward does not make an already-used test set independent again. Complete the repair dictionary and write a limitations statement before handing in.

In [ ]:
# YOUR CODE: repair each boundary using the four phrases in expected_repairs below.
repairs = None
# YOUR CODE: write a limitations statement covering the historical data, category codes,
# threshold, and review use—not automatic decisions. Include the phrase 'not automatic'.
limitations = ""
expected_repairs = {"adapt": "training only", "candidate": "fresh model optimizer callback", "select": "validation loss", "used_test": "new independent evaluation set"}
print("Workflow repairs:", repairs)
print("Limitations:", limitations)
check("workflow_debugging", repairs == expected_repairs and isinstance(limitations, str) and all(word in limitations.lower() for word in ["historical", "category", "threshold", "review", "not automatic"]), "Use the four repair phrases shown in expected_repairs, then explain historical data, numeric category codes, the threshold and review rather than automatic decisions.")

## Hand-in

In [ ]:
# Hand-in — run this last, then paste the line it prints into the lesson page
handin()